In [1]:
import pandas as pd 
import numpy as np 

In [3]:
results = pd.read_parquet('../data/interim/results_clean.parquet')
print(results.head())

        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3 1875-03-06   England  Scotland           2           2   Friendly   London   
4 1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  


In [5]:
# Same data scope + split as Model 1, so both models are evaluated on the identical eval set
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)

# Leakage-safe split — boundaries mutually exclusive so no match is in two slices
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)

train_df = model_df[model_df['date'] < '2024-01-01']                   # Elo burns in on this
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]  # held-out test set (WC excluded)
wc2026_df = model_df[wc2026_mask]                                      # forecast target, held aside

print(f"train {len(train_df)} | eval {len(eval_df)} | wc2026 {len(wc2026_df)}")

train 38900 | eval 2543 | wc2026 79


In [8]:
def compute_elo_ratings(matches, k=20, base_rating=1500):
    ratings = {}   # team -> current rating (new teams start at base_rating)

    # Process matches in chronological order, nudging ratings after each game
    for _, row in matches.iterrows():
        home, away = row['home_team'], row['away_team']
        rating_home = ratings.get(home, base_rating)
        rating_away = ratings.get(away, base_rating)

        # Expected home score from the rating gap (S-curve, 0..1)
        expected_home = 1 / (1 + 10 ** ((rating_away - rating_home) / 400))

        # Actual home result: 1 win, 0.5 draw, 0 loss
        if row['home_score'] > row['away_score']:
            actual_home = 1.0
        elif row['home_score'] == row['away_score']:
            actual_home = 0.5
        else:
            actual_home = 0.0

        # Nudge both ratings by K x (surprise); zero-sum between the two teams
        change = k * (actual_home - expected_home)
        ratings[home] = rating_home + change
        ratings[away] = rating_away - change

    return ratings


# Sanity check: build ratings on train, inspect the top
elo_ratings = compute_elo_ratings(train_df)
for team, rating in sorted(elo_ratings.items(), key=lambda item: item[1], reverse=True)[:10]:
    print(f"{team:20s} {rating:.0f}")

Argentina            1973
France               1937
Brazil               1930
Spain                1916
England              1909
Belgium              1896
Portugal             1889
Netherlands          1877
Colombia             1877
Uruguay              1855
